In [9]:
!pip install numdifftools
import numpy as np
import pandas as pd
import math
import numdifftools as nd
import scipy as sc
from math import log
from scipy.special import gammaln
from scipy.optimize import minimize
from scipy.optimize._numdiff import approx_derivative


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
# data = data = pd.read_excel(r"C:\Users\Amira\Downloads\techstocks.xlsx")
data = pd.read_excel("data/techstocks.xlsx")

In [11]:
data

,date,AAPL,MSFT,AMZN,GOOG,TSLA
0,2023-07-03,-0.781516,-0.751633,-0.107452,-0.339512,6.668035
1,2023-07-05,-0.588856,0.047341,0.122796,1.702412,0.946123
2,2023-07-06,0.250554,0.918448,-1.561448,-1.395979,-2.125228
3,2023-07-07,-0.590882,-1.193858,1.100188,-0.655407,-0.765931
4,2023-07-10,-1.091514,-1.611290,-2.063054,-2.759554,-1.771978
...,...,...,...,...,...,...
622,2025-12-23,0.511661,0.397212,1.611085,1.387562,-0.650735
623,2025-12-24,0.530969,0.240029,0.103335,-0.003161,-0.032958
624,2025-12-26,-0.149848,-0.063542,0.060228,-0.225179,-2.125855
625,2025-12-29,0.131595,-0.125150,-0.193718,-0.181132,-3.327112


In [12]:
def MVT_MLE_approach2(X):
    n, p = X.shape
    m = np.zeros(p) # initial mu
    v = np.log(5) # initial value for log(nu-2)
    S = np.eye(p) # initial Sigma
    s_vech = S[np.tril_indices(S.shape[0])] # p(p+1)/2 uniques params - lower matrix
    theta_initial = np.append(np.array(m), np.append(s_vech, v))

    mle = sc.optimize.minimize(
        LL_mvt_appr,
        theta_initial,
        args=(X,),
        method="L-BFGS-B",
        options={'disp': False, 'maxiter': 2500}
    )
    converged = mle['success']
    Theta_tilde = mle.x

    Hessfun = nd.Hessian(LL_mvt_appr)
    H = Hessfun(Theta_tilde, X)
    Hinv = np.linalg.pinv(H)
    V_asy = Hinv / n # asymptotic variance (/n because average)
    V_sam = Hinv # sample variance

    Theta = g_fun_mvt(Theta_tilde, p).reshape(-1, 1)
    Jfun = nd.Jacobian(g_fun_mvt)
    D = Jfun(Theta_tilde, p) # jacobian of g()
    VCV_asy = np.squeeze(D @ V_asy @ D.T) # asy variance theta
    VCV_sam = np.squeeze(D @ V_sam @ D.T) # sample variance theta

    return Theta, np.squeeze(VCV_sam), np.squeeze(VCV_asy), converged


def g_fun_mvt(pars, p):
    m = pars[0:p]
    v = 2 + np.exp(pars[-1]) # enforce nu > 2 (nu = 2 + exp(.))

    # put scale elements in lower triangular matrix L
    a = pars[p:-1]
    L = np.zeros([p, p])
    idx = np.tril_indices(p)
    L[idx] = a

    S = L @ L.T # calculate Sigma
    b = S[np.tril_indices(S.shape[0])] # extract unique elements from Sigma
    Theta = np.append(np.array(m), np.append(b, v))
    return Theta


def LL_mvt_appr(pars, X):
    n, p = X.shape
    # g() : translate theta_tilde to theta
    m = np.reshape(pars[0:p], (-1, 1))
    v = 2 + np.exp(pars[-1]) # enforce nu > 2

    a = pars[p:-1]
    L = np.zeros([p, p])
    idx = np.tril_indices(p)
    L[idx] = a

    S = L @ L.T

    # fill into and evaluate the LL
    nl = negloglik_mvt(X, n, p, m, S, v)
    return np.array([nl / n])


def negloglik_mvt(X, n, p, m, S, v):
    Si = np.linalg.pinv(S)
    ll1 = n * (sc.special.gammaln((v + p) / 2)
               - sc.special.gammaln(v / 2)
               - (p / 2) * np.log(v * np.pi)
               - 0.5 * np.log(np.linalg.det(S)))

    xd = X - np.ones([n, 1]) @ m.T # n x p matrix
    ll2 = -0.5 * (v + p) * np.log(1 + np.einsum('ij,ij->i', xd @ Si, xd) / v).sum()

    nll = -(ll1 + ll2) # minimizer = negative LL
    return nll

X_df = data[["AAPL", "MSFT", "AMZN", "GOOG", "TSLA"]]
X = X_df.values

Theta, VCV_sam, VCV_asy, converged = MVT_MLE_approach2(X)
print("Converged:", converged)    

def defining_theta(theta, p):
    theta = np.asarray(theta).ravel()
    mu = theta[:p]
    nu = theta[-1]
    a = theta[p:-1]

    Sigma = np.zeros((p, p))
    idx = np.tril_indices(p)
    Sigma[idx] = a
    Sigma = Sigma + Sigma.T - np.diag(np.diag(Sigma))
    return mu, Sigma, nu

def report_mvt_results(theta_hat, VCV_asy, X, names=None):
    n, p = X.shape
    if names is None:
        names = [f"X{j+1}" for j in range(p)]

    se = np.sqrt(np.diag(VCV_asy))

    mu_hat, Sigma_hat, nu_hat = defining_theta(theta_hat, p)

    se_mu = se[:p]
    se_nu = se[-1]

    cov_implied = (nu_hat / (nu_hat - 2.0)) * Sigma_hat
    vol = np.sqrt(np.diag(cov_implied))

    corr_implied = cov_implied / np.outer(vol, vol)
    
    C = corr_implied.copy()
    np.fill_diagonal(C, -np.inf)
    i, j = np.unravel_index(np.argmax(C), C.shape)

    print("\nMVT MLE estimates (with asymptotic SE)")
    for nm, m, s in zip(names, mu_hat, se_mu):
        print(f"{nm:5s}: mu_hat = {m: .6f}   SE = {s: .6f}   t = {m/s: .3f}")

    print(f"\nnu_hat = {nu_hat:.6f}   SE = {se_nu:.6f}   t = {nu_hat/se_nu: .3f}")

    print("\nImplied daily volatility (from Cov = nu/(nu-2)*Sigma)")
    for nm, v in zip(names, vol):
        print(f"{nm:5s}: {v:.4f}%")

    print("\nImplied correlation matrix (rounded)")
    print(np.round(corr_implied, 3))

    print(f"\nHighest correlation pair: {names[i]} - {names[j]} = {corr_implied[i, j]:.3f}")

    se_sig_vech = se[p:-1]
    print("\nSigma_hat (scale matrix in MVT)")
    print(Sigma_hat)

    print("\nSEs for vech(Sigma) (lower triangle incl diag)")
    k = 0
    for r in range(p):
        for c in range(r + 1):
            print(f"Sigma[{r+1},{c+1}] = {Sigma_hat[r,c]: .6f}   SE = {se_sig_vech[k]: .6f}")
            k += 1

    return {
        "mu_hat": mu_hat,
        "Sigma_hat": Sigma_hat,
        "nu_hat": nu_hat,
        "VCV_asy": VCV_asy,
        "se": se,
        "cov_implied": cov_implied,
        "corr_implied": corr_implied,
        "vol": vol,
    }

names = ["AAPL", "MSFT", "AMZN", "GOOG", "TSLA"]
out = report_mvt_results(Theta, VCV_asy, X, names=names)

/var/folders/2y/nsymfvp93951y3bnw1y4d2lh0000gn/T/ipykernel_9990/2821939138.py:9: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  mle = sc.optimize.minimize(


Converged: True

MVT MLE estimates (with asymptotic SE)
AAPL : mu_hat =  0.105645   SE =  0.051964   t =  2.033
MSFT : mu_hat =  0.115758   SE =  0.045991   t =  2.517
AMZN : mu_hat =  0.096148   SE =  0.064306   t =  1.495
GOOG : mu_hat =  0.196546   SE =  0.060186   t =  3.266
TSLA : mu_hat =  0.071160   SE =  0.130871   t =  0.544

nu_hat = 4.354238   SE = 0.372494   t =  11.689

Implied daily volatility (from Cov = nu/(nu-2)*Sigma)
AAPL : 1.6078%
MSFT : 1.4152%
AMZN : 1.9777%
GOOG : 1.8539%
TSLA : 4.0261%

Implied correlation matrix (rounded)
[[1.    0.462 0.43  0.429 0.388]
 [0.462 1.    0.642 0.531 0.395]
 [0.43  0.642 1.    0.585 0.411]
 [0.429 0.531 0.585 1.    0.41 ]
 [0.388 0.395 0.411 0.41  1.   ]]

Highest correlation pair: MSFT - AMZN = 0.642

Sigma_hat (scale matrix in MVT)
[[1.39774091 0.56892815 0.73904302 0.69108662 1.35710245]
 [0.56892815 1.08287003 0.97159197 0.75295692 1.21729863]
 [0.73904302 0.97159197 2.11469641 1.15939512 1.76871973]
 [0.69108662 0.75295692 1.1